[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/03-OCRfacturas.ipynb)


# 03 — Comparando métodos de OCR: de lo clásico a un modelo de Hugging Face

En **`02-PDF_reporte`** vimos que `pypdf` solo sirve si el PDF ya trae texto seleccionable. En cuanto el "documento" es en realidad una foto, `extract_text()` regresa (casi) nada — la información sigue ahí, pero como **píxeles**, no como caracteres.

Aquí comparamos **cuatro formas distintas** de resolver eso sobre el mismo problema real: **extraer los datos de una factura** (folio, fecha, emisor, total) a partir de su imagen.

| Ronda | Método | Qué es |
|---|---|---|
| 1 | **Tesseract** | El motor de OCR "clásico", sin deep learning. |
| 2 | **EasyOCR** | OCR con redes neuronales, entrenado para texto "del mundo real". |
| 3 | **PaddleOCR** | OCR moderno, más robusto con layouts distintos. |
| 4 | **[Falcon-OCR](https://huggingface.co/tiiuae/Falcon-OCR)** | Modelo de Hugging Face especializado en documentos (300M parámetros): entiende que una tabla es una tabla y te la regresa ya estructurada. |

Corremos las cuatro rondas sobre **las mismas 3 facturas** y comparamos contra el dato real. Al final, con Falcon-OCR, probamos algo más: **¿importa cómo capturaste el documento?** (PDF renderizado limpio vs. una foto del PDF).


## Carpeta `public/`

Igual que en `02-PDF_reporte`, este notebook espera una carpeta **`public/`** junto al notebook con los archivos que se usan.

Las 3 facturas (`factura_1.pdf`, `factura_2.pdf`, `factura_3.pdf`) son **ejemplos inventados** para esta clase — mismo formato que un CFDI mexicano real (folio, RFC emisor/receptor, conceptos, IVA, total, sello y QR de adorno), pero con nombres, RFC y montos completamente ficticios, y como PDF (así es como sale una factura real). Como son nuestras, ya vienen en el repo. Cada OCR necesita una imagen, así que la primera celda de código convierte cada PDF a imagen una sola vez.

**En Colab:** crea la carpeta `public` (panel de archivos → clic derecho → *Nueva carpeta*) y arrastra ahí `factura_1.pdf`, `factura_2.pdf`, `factura_3.pdf`, `ey_100_casos_rentables_ia_2026.pdf` y `fakeine.jpeg` (descárgalos de la carpeta `modulo_4/public/` del repo en GitHub).

**En local:** si clonaste el repo, `modulo_4/public/` ya trae los 5 archivos.


In [ ]:
%pip install -q pypdfium2

from pathlib import Path

import pypdfium2 as pdfium
from PIL import Image

CARPETA = Path("public")
RUTAS_FACTURAS = {
    "factura_1.pdf": CARPETA / "factura_1.pdf",
    "factura_2.pdf": CARPETA / "factura_2.pdf",
    "factura_3.pdf": CARPETA / "factura_3.pdf",
}


def pdf_a_imagen(ruta_pdf):
    return pdfium.PdfDocument(str(ruta_pdf))[0].render(scale=2).to_pil().convert("RGB")


# Convertimos una sola vez y reutilizamos la misma imagen en las 4 rondas.
IMAGENES_FACTURAS = {nombre: pdf_a_imagen(ruta) for nombre, ruta in RUTAS_FACTURAS.items()}


## El dato real

Como armamos nosotros las facturas, sabemos exactamente qué debería salir. Guardamos aquí los 4 campos correctos de cada una para comparar automáticamente al final, en vez de "eyeballear".


In [ ]:
DATO_REAL = {
    "factura_1.pdf": {
        "folio": "001",
        "fecha": "2026-03-14",
        "emisor": "Laura Ximena Reyes Cortés",
        "total": "15660.00",
    },
    "factura_2.pdf": {
        "folio": "002",
        "fecha": "2026-05-02",
        "emisor": "Soluciones Digitales del Bajío",
        "total": "9512.00",
    },
    "factura_3.pdf": {
        "folio": "003",
        "fecha": "2026-07-21",
        "emisor": "Miguel Ángel Torres Domínguez",
        "total": "25520.00",
    },
}


## Ronda 1 — Tesseract

**Tesseract** no usa deep learning: detecta formas de caracteres contra patrones. Es el motor de OCR "de toda la vida".

**Ventajas:** gratis, instantáneo, no necesita GPU ni internet (una vez instalado).

**Desventajas:** solo texto plano, sin noción de estructura (tablas, columnas); le cuesta con fotos giradas o de mala calidad; tú tienes que armar los campos con regex.


In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
%pip install -q pytesseract


In [ ]:
import time

import pytesseract

resultados_tesseract = {}
for nombre, imagen in IMAGENES_FACTURAS.items():
    t0 = time.time()
    texto = pytesseract.image_to_string(imagen)
    resultados_tesseract[nombre] = {"texto": texto, "segundos": time.time() - t0}

for nombre, r in resultados_tesseract.items():
    print(f"--- {nombre} ({r['segundos']:.2f}s) ---")
    print(r["texto"][:400])
    print()


## De texto plano a campos: una regex mínima

Las 3 facturas usan **la misma plantilla** (nosotros la armamos), así que cuatro líneas de regex bastan para sacar el folio, la fecha, el emisor y el total de cualquier texto plano: buscamos la etiqueta (`"Folio:"`, `"Total:"`...) y capturamos lo que sigue.


In [ ]:
import re


def buscar(patron, texto):
    coincidencia = re.search(patron, texto)
    return coincidencia.group(1).strip() if coincidencia else None


def extraer_campos(texto):
    # "Subtotal" tambien contiene "total", asi que nos quedamos con la
    # ULTIMA coincidencia (el Total real siempre viene despues).
    totales = re.findall(r"[TtOo]otal:?\s*\$?\s*([\d.,]+)", texto)
    return {
        "folio": buscar(r"Folio:?\s*(\S+)", texto),
        "fecha": buscar(r"emisi.n:?\s*(\d{4}-\d{2}-\d{2})", texto),
        "emisor": buscar(r"[Nn]ombre emisor:?\s*(.+)", texto),
        "total": totales[-1] if totales else None,
    }


## Función para comparar contra el dato real

La reutilizamos para las 4 rondas.


In [ ]:
import unicodedata

import pandas as pd


def normalizar(valor):
    if valor is None:
        return ""
    # NFKD + encode a ascii quita acentos (bajío -> bajio) para que
    # el OCR (que a veces se los come) siga contando como acierto.
    texto = unicodedata.normalize("NFKD", str(valor).lower())
    texto = texto.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]", "", texto)


def campo_correcto(extraido, real):
    return normalizar(real) != "" and normalizar(real) in normalizar(extraido)


def evaluar_metodo(nombre_metodo, resultados, extractor):
    """resultados: {nombre_factura: {'texto': ..., 'segundos': ...}}"""
    filas = []
    for nombre_factura, r in resultados.items():
        real = DATO_REAL[nombre_factura]
        campos = extractor(r)
        fila = {"metodo": nombre_metodo, "factura": nombre_factura, "segundos": round(r["segundos"], 2)}
        for campo in ("folio", "fecha", "emisor", "total"):
            fila[campo] = "✅" if campo_correcto(campos.get(campo), real[campo]) else "❌"
        filas.append(fila)
    return pd.DataFrame(filas)


In [ ]:
tabla_tesseract = evaluar_metodo(
    "Tesseract", resultados_tesseract, lambda r: extraer_campos(r["texto"])
)
tabla_tesseract


## Ronda 2 — EasyOCR

OCR con redes neuronales, entrenado para texto "del mundo real" (facturas, tickets, letreros). Mismo procedimiento, mismas 3 facturas, mismo extractor de campos — solo cambia el motor.

**Ventajas:** mejor que Tesseract con fotos reales (ángulos, fondos, fuentes variadas).

**Desventajas:** más lento, descarga un modelo de ~500 MB la primera vez, y sigue sin entender que hay una tabla — el mismo problema de columnas que Tesseract.


In [ ]:
%pip install -q easyocr


In [ ]:
import numpy as np
import easyocr

lector_easyocr = easyocr.Reader(["es"], gpu=False)

resultados_easyocr = {}
for nombre, imagen in IMAGENES_FACTURAS.items():
    t0 = time.time()
    lineas = lector_easyocr.readtext(np.array(imagen), detail=0)
    resultados_easyocr[nombre] = {"texto": "\n".join(lineas), "segundos": time.time() - t0}

tabla_easyocr = evaluar_metodo(
    "EasyOCR", resultados_easyocr, lambda r: extraer_campos(r["texto"])
)
tabla_easyocr


## Ronda 3 — PaddleOCR

Otro motor con deep learning, con mejor manejo de rotación y de layouts variados que EasyOCR.

**Ventajas:** más robusto ante fotos giradas o con layouts distintos entre sí.

**Desventajas:** instalación más pesada; con el modo simple que usamos aquí, tampoco resuelve tablas complejas (para eso existe **PP-StructureV3**, que ya reconstruye tablas completas — vale la pena explorarlo si quieren ir más lejos).


In [ ]:
%pip install -q paddlepaddle paddleocr


In [ ]:
from paddleocr import PaddleOCR

# Si tu version de paddleocr ya no acepta use_angle_cls/lang asi, revisa
# la documentacion actual del paquete instalado (la API ha cambiado entre versiones).
lector_paddle = PaddleOCR(use_angle_cls=True, lang="es")

resultados_paddle = {}
for nombre, imagen in IMAGENES_FACTURAS.items():
    t0 = time.time()
    resultado = lector_paddle.ocr(np.array(imagen), cls=True)
    texto = "\n".join(linea[1][0] for bloque in resultado for linea in bloque)
    resultados_paddle[nombre] = {"texto": texto, "segundos": time.time() - t0}

tabla_paddle = evaluar_metodo(
    "PaddleOCR", resultados_paddle, lambda r: extraer_campos(r["texto"])
)
tabla_paddle


## Ronda 4 — Falcon-OCR, un modelo de Hugging Face

Hasta aquí sacamos **texto plano** y lo peinamos con regex. **[Falcon-OCR](https://huggingface.co/tiiuae/Falcon-OCR)** (de TII) es distinto: es un modelo chico (300M parámetros, ~3× más chico que otros modelos de su categoría) **especializado en documentos**, no un chatbot genérico. Sabe hacer tres cosas según se lo pidas: texto plano, fórmulas en LaTeX, o **tablas en HTML ya estructuradas**.

**Necesita GPU.** Antes de seguir: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

Vamos a hacer exactamente lo que harían en su trabajo: entrar a la página del modelo en Hugging Face, copiar el código de uso rápido ("Quickstart") de la ficha del modelo, y pegarlo aquí. Es este:


In [ ]:
%pip install -q -U transformers accelerate


In [ ]:
# Codigo de la ficha del modelo (https://huggingface.co/tiiuae/Falcon-OCR), copiado y pegado casi tal cual
# -- solo cambiamos la imagen por una de nuestras facturas (ya convertida de PDF a imagen arriba).
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/Falcon-OCR",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

image = IMAGENES_FACTURAS["factura_1.pdf"]
texts = model.generate(image)  # categoria por default: texto plano
print(texts[0])


Eso es todo lo que pedía la ficha del modelo. Con el modelo ya cargado en memoria (`model`), lo corremos sobre nuestras 3 facturas y reutilizamos el mismo `extraer_campos` de las rondas anteriores.


In [ ]:
resultados_falcon = {}
for nombre, imagen in IMAGENES_FACTURAS.items():
    t0 = time.time()
    texto = model.generate(imagen)[0]
    resultados_falcon[nombre] = {"texto": texto, "segundos": time.time() - t0}

tabla_falcon = evaluar_metodo(
    "Falcon-OCR", resultados_falcon, lambda r: extraer_campos(r["texto"])
)
tabla_falcon


### El truco: pedirle la tabla directo

Falcon-OCR también sabe leer `category="table"` y regresar la tabla de renglones **ya en HTML**, en vez de texto plano donde las columnas se pueden revolver. Probémoslo con la factura que tiene más conceptos (`factura_3.jpg`, 4 renglones).


In [ ]:
from IPython.display import HTML, display

tabla_html = model.generate(IMAGENES_FACTURAS["factura_3.pdf"], category="table")[0]
print(tabla_html)
display(HTML(tabla_html))


Esa es la ventaja real de un modelo especializado en documentos: no tuvimos que escribir ni una regex para reconstruir la tabla — se la pedimos directo.

**Ventajas:** entiende que una tabla es una tabla (HTML estructurado, sin regex); modelo chico (300M) y especializado, no un LLM genérico de propósito general.

**Desventajas:** necesita GPU — no es viable en CPU; no es conversacional, solo tiene 3 modos fijos (texto/fórmula/tabla), no le puedes "preguntar" cosas libres como a un chatbot.


### ¿Importa cómo capturaste el documento? PDF renderizado vs. foto del PDF

Todo lo anterior fue con **fotos ya digitales** (nuestras facturas de ejemplo). Pero en la vida real, muchas veces el punto de partida es un **PDF** (como el de `02-PDF_reporte`) y alguien lo **imprime y le toma una foto con el celular** en vez de mandar el archivo digital.

Corremos **Falcon-OCR** sobre **la misma página**, capturada de dos formas:

1. **PDF renderizado limpio** — convertimos la página del PDF a imagen directamente.
2. **"Foto del PDF"** — simulamos una foto de esa misma página ya impresa: ángulo, sombra/reflejo, menos resolución, compresión JPEG.


In [ ]:
pagina_limpia = pdf_a_imagen("public/ey_100_casos_rentables_ia_2026.pdf")
pagina_limpia


In [ ]:
import numpy as np


def simular_foto(imagen, angulo=6, escala=0.5, calidad_jpeg=40):
    """Aproxima una foto de celular a un documento impreso: rotación, menos resolución,
    sombra/gradiente de luz y compresión JPEG agresiva."""
    from io import BytesIO

    img = imagen.rotate(angulo, expand=True, fillcolor=(255, 255, 255))
    nuevo_ancho = int(img.width * escala)
    nuevo_alto = int(img.height * escala)
    img = img.resize((nuevo_ancho, nuevo_alto))

    # Gradiente de sombra/reflejo de izquierda a derecha
    gradiente = np.tile(np.linspace(0.65, 1.0, nuevo_ancho), (nuevo_alto, 1))
    arreglo = np.array(img).astype(float)
    for canal in range(3):
        arreglo[:, :, canal] *= gradiente
    img = Image.fromarray(np.clip(arreglo, 0, 255).astype("uint8"))

    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=calidad_jpeg)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


pagina_foto = simular_foto(pagina_limpia)
pagina_foto


In [ ]:
texto_limpio_pdf = model.generate(pagina_limpia)[0]
texto_foto_pdf = model.generate(pagina_foto)[0]

print("--- PDF renderizado limpio ---")
print(texto_limpio_pdf[:400])
print()
print("--- Foto simulada del PDF ---")
print(texto_foto_pdf[:400])


Si la segunda transcripción tiene más errores, palabras cortadas o directamente le faltan pedazos, esa es la lección: **el modelo importa, pero la calidad de la captura del documento importa tanto o más.** Es la misma razón por la que, en producción, casi siempre conviene pedir el PDF/archivo original en vez de aceptar una foto.


## Comparando todo

Juntamos las tablas de las 4 rondas.


In [ ]:
comparacion = pd.concat(
    [tabla_tesseract, tabla_easyocr, tabla_paddle, tabla_falcon], ignore_index=True
)
comparacion


In [ ]:
resumen = comparacion.groupby("metodo").agg(
    segundos_promedio=("segundos", "mean"),
    aciertos_folio=("folio", lambda s: (s == "✅").sum()),
    aciertos_fecha=("fecha", lambda s: (s == "✅").sum()),
    aciertos_emisor=("emisor", lambda s: (s == "✅").sum()),
    aciertos_total=("total", lambda s: (s == "✅").sum()),
)
resumen


## Cierre

Cada ronda tuvo su tabla de ventajas/desventajas arriba — el resumen: entre más entiende el modelo de **estructura** (tabla, layout), menos regex necesitas escribir tú. Y no menos importante: **la captura del documento importa tanto como el método** — un PDF limpio y una foto borrosa del mismo PDF no le dan la misma información a ningún método.

En producción, muchas empresas ya resuelven esto con **servicios administrados** (Google Document AI, AWS Textract, Azure Document Intelligence) que ya vienen entrenados para facturas — la ventaja de hacerlo "a mano" aquí es entender **qué está pasando por dentro** antes de delegarlo a una caja negra.


## Bonus opcional — otro tipo de documento: una identificación

*(Sáltate esta sección si vas corto de tiempo — es solo para mostrar que la misma técnica generaliza a otro tipo de documento, no solo facturas.)*

Reutilizamos el modelo ya cargado (Falcon-OCR) sobre una credencial **de ejemplo, generada para fines didácticos** (`public/fakeine.jpeg`) — no es un documento real.


In [ ]:
texto_ine = model.generate(Image.open("public/fakeine.jpeg"))[0]
print(texto_ine)


Aquí no hay una plantilla fija como en las facturas, así que ya no alcanza con el `extraer_campos` de antes — tocaría escribir patrones nuevos para nombre, CURP, fecha de nacimiento, etc. Con Tesseract o EasyOCR sería el mismo trabajo extra. Falcon-OCR al menos ya te dio el texto limpio y ordenado como punto de partida.
